# DINO Patch Token 设计 PoC (A3.4 / A3.5)

对应计划 §A3.4 / §A3.5：验证两件事

1. **patch token 的语义意义**：每格向量与 CLS 的空间耦合、七类场景的可视化对比。
2. **下采样到 16×16 + INT8 量化的信息损失**：PCA 三通道可视化 / cosine 保持率应 ≥ 0.999。

运行前请先按 §A2-M1 导出好 ONNX（`PhotoViewer/Assets/Models/dinov3_vits16.onnx`，双输出）。

---

**依赖**：`pip install onnxruntime numpy pillow matplotlib scikit-learn`

In [ ]:
from pathlib import Path
import numpy as np
import onnxruntime as ort
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# --- 配置 ---
ONNX_PATH = Path('PhotoViewer/Assets/Models/dinov3_vits16.onnx')
INPUT_SIZE = 518
PATCH_SIZE = 16
NATIVE_GRID = INPUT_SIZE // PATCH_SIZE   # 518/16 = 32
TARGET_GRID = 16                         # 与 CV 网格对齐
POOL = NATIVE_GRID // TARGET_GRID        # 32 → 16 需 2×2 avg pool

# ImageNet 归一化（与 C# 推理侧一致）
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# TODO: 填 §0.3 七类场景各 1~2 张
SAMPLES = [
    # Path(r'd:/path/to/sample1.jpg'),
]

In [ ]:
sess = ort.InferenceSession(str(ONNX_PATH), providers=['CPUExecutionProvider'])
print('inputs :', [i.name for i in sess.get_inputs()])
print('outputs:', [o.name for o in sess.get_outputs()])

def preprocess(path: Path) -> np.ndarray:
    img = Image.open(path).convert('RGB').resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)
    a = np.asarray(img, dtype=np.float32) / 255.0
    a = (a - MEAN) / STD
    return a.transpose(2, 0, 1)[None, ...].astype(np.float32)

def run(path: Path):
    cls, patch = sess.run(['cls_embedding', 'patch_tokens'], {'pixel_values': preprocess(path)})
    # patch: [1, N*N, D] → [N, N, D]
    return cls[0], patch[0].reshape(NATIVE_GRID, NATIVE_GRID, -1)

In [ ]:
def avg_pool_2d(grid: np.ndarray, pool: int) -> np.ndarray:
    """整数倍 2D avg pool：[Gn, Gn, D] → [Gt, Gt, D]。"""
    Gn = grid.shape[0]
    assert Gn % pool == 0, (Gn, pool)
    Gt = Gn // pool
    D = grid.shape[-1]
    return grid.reshape(Gt, pool, Gt, pool, D).mean(axis=(1, 3))

def quantize_int8_per_token(grid: np.ndarray):
    """每个 token 独立 min-max → INT8；返回 (int8, scale, zero_point)。"""
    G, _, D = grid.shape
    flat = grid.reshape(-1, D)               # [G*G, D]
    mn = flat.min(axis=1, keepdims=True)
    mx = flat.max(axis=1, keepdims=True)
    scale = (mx - mn) / 255.0
    scale = np.where(scale > 0, scale, 1.0)
    zp = np.round(-mn / scale).astype(np.int32)
    q = np.clip(np.round(flat / scale) + zp, 0, 255).astype(np.uint8)
    return q.reshape(G, G, D), scale.reshape(G, G), zp.reshape(G, G).astype(np.int32)

def dequantize_int8_per_token(q, scale, zp):
    return ((q.astype(np.float32) - zp[..., None]) * scale[..., None]).astype(np.float32)

def cosine(a, b, eps=1e-12):
    a = a / (np.linalg.norm(a, axis=-1, keepdims=True) + eps)
    b = b / (np.linalg.norm(b, axis=-1, keepdims=True) + eps)
    return (a * b).sum(axis=-1)

In [ ]:
def show_pca_rgb(ax, grid2d: np.ndarray, title: str):
    """把 [G, G, D] 通过 PCA 降到 3 通道当 RGB 可视化。"""
    G = grid2d.shape[0]
    flat = grid2d.reshape(-1, grid2d.shape[-1])
    pc = PCA(n_components=3).fit_transform(flat)
    pc = (pc - pc.min(0)) / (pc.max(0) - pc.min(0) + 1e-12)
    ax.imshow(pc.reshape(G, G, 3))
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])

def evaluate(path: Path):
    cls, native = run(path)
    pooled = avg_pool_2d(native, POOL)                   # 32×32 → 16×16
    q, scale, zp = quantize_int8_per_token(pooled)
    restored = dequantize_int8_per_token(q, scale, zp)

    # INT8 保持率：每格原向量 vs 解量化向量
    cos = cosine(
        pooled.reshape(-1, pooled.shape[-1]),
        restored.reshape(-1, restored.shape[-1]),
    )

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(Image.open(path).resize((256, 256)))
    axes[0].set_title(path.name); axes[0].set_xticks([]); axes[0].set_yticks([])
    show_pca_rgb(axes[1], native,   f'native {NATIVE_GRID}×{NATIVE_GRID}')
    show_pca_rgb(axes[2], pooled,   f'pooled {TARGET_GRID}×{TARGET_GRID}')
    show_pca_rgb(axes[3], restored, f'int8 restored (cos_min={cos.min():.4f}, cos_mean={cos.mean():.4f})')
    plt.show()
    return {
        'cos_min': float(cos.min()),
        'cos_mean': float(cos.mean()),
        'cls_norm': float(np.linalg.norm(cls)),
    }

results = {p.name: evaluate(Path(p)) for p in SAMPLES}
results

## 验收判断

- **下采样可视化**：`native` 与 `pooled` 的 PCA-RGB 语义布局应大致一致（人/天/物体区块颜色类别保留）。
- **INT8 保持率**：`cos_min` 应 ≥ 0.999；批次平均 `cos_mean` 记录到 §A3.6 验收 checklist。
- 若发现 INT8 cos_min 明显 < 0.999 → 退到 f16（§A3.4 表格的 2× 压缩）；本 notebook 已预留该分支，修改 `quantize_int8_per_token` 即可。